# Week 16: Distributed Training & Model Optimization
## Notebook 08: Fully Sharded Data Parallel (FSDP) & Hybrid Sharding (HSDP)

### 1. Conceptual Intuition & Theoretical Background
Standard Distributed Data Parallel (DDP) duplicates all model weights, optimizer states, and gradients on every GPU, limiting maximum model size to single-GPU VRAM.

**Fully Sharded Data Parallel (FSDP / ZeRO-3)** shards all three components across all $N$ GPUs:
$$\text{Memory per GPU} = \frac{\text{Params} + \text{Gradients} + \text{Optimizer States}}{N}$$
- **Forward Pass:** Broadcasts layer weights via an **All-Gather** collective just before execution, then frees them immediately.
- **Backward Pass:** Re-gathers weights via **All-Gather**, computes gradients, and reduces them across ranks via **Reduce-Scatter**.

**Hybrid Sharding Data Parallel (HSDP):**
When training across multi-node clusters, inter-node network bandwidth (e.g. 100Gbps RoCE / InfiniBand) is much slower than intra-node NVLink (900GB/s).
- **HSDP** applies **Full Sharding (FSDP) within each node** (across the 8 NVLink GPUs).
- Applies standard **DDP across nodes**, avoiding cross-node All-Gather bottlenecks.

---

### 2. Scratch Python Implementation: FSDP Parameter Shard Reconstitution

In [ ]:
import numpy as np

class MockFSDPShardManager:
    def __init__(self, full_weight_shape, world_size=4):
        self.world_size = world_size
        self.full_shape = full_weight_shape
        self.flat_dim = int(np.prod(full_weight_shape))
        
        # Partition flat dimension into equal slices
        self.shard_dim = self.flat_dim // world_size
        
        # Initialize full master tensor
        np.random.seed(42)
        self.master_weights = np.random.normal(0, 0.02, self.flat_dim)
        
        # Each GPU stores only its individual shard (1 / world_size)
        self.gpu_shards = [
            self.master_weights[r * self.shard_dim : (r + 1) * self.shard_dim].copy()
            for r in range(world_size)
        ]
        
    def simulate_forward_allgather(self, rank):
        """Simulates all-gather collective on rank: rebuilds full weights temporarily."""
        gathered_buffer = np.zeros(self.flat_dim)
        for r in range(self.world_size):
            gathered_buffer[r * self.shard_dim : (r + 1) * self.shard_dim] = self.gpu_shards[r]
            
        full_tensor = gathered_buffer.reshape(self.full_shape)
        return full_tensor
        
    def simulate_deallocation_after_layer(self, full_tensor):
        """Immediately deallocate full weights to save VRAM."""
        del full_tensor
        return None

# Test Sharded Model Allocation
world_size = 4
weight_shape = (1024, 1024) # 1 Million params
manager = MockFSDPShardManager(weight_shape, world_size=world_size)

print(f"Full Model Parameter Count:   {manager.flat_dim}")
print(f"Memory Sharded per GPU:       {manager.shard_dim} parameters (1/{world_size}th)")

# Rank 0 executes layer forward
reconstructed_layer = manager.simulate_forward_allgather(rank=0)
print(f"Rank 0 Reconstructed Tensor:  {reconstructed_layer.shape}")

# Forward completed: deallocate
reconstructed_layer = manager.simulate_deallocation_after_layer(reconstructed_layer)
print("Reconstructed parameter tensor deallocated. Peak memory returned to base shard size!")

### 3. Step-by-Step Code Walkthrough
1. **1/N Sharding**: Only a slice is retained in physical memory when layer is inactive.
2. **On-Demand All-Gather**: Full weights are gathered just-in-time for forward/backward computation.
3. **Immediate Free**: Deallocates full tensors immediately to keep memory bounded.

---

### 4. Industry Applications & Trade-offs
- **Training 70B+ LLMs without Tensor Parallelism:** FSDP enables training large architectures using standard PyTorch primitives.
- **Trade-off:** $50\%$ higher communication overhead due to forward All-Gather operations.

---

### 5. Top Interview Questions & Answers
**Q1: When should an engineer use HSDP (Hybrid Sharding) instead of pure FSDP?**
> **Answer:** In multi-node clusters where inter-node network bandwidth is a bottleneck. HSDP limits expensive All-Gather operations to high-speed intra-node NVLink, using standard DDP gradient synchronization across nodes.

**Q2: What is the exact communication overhead difference between DDP and FSDP?**
> **Answer:** DDP performs a single All-Reduce ($2 \times \text{Params}$ communication volume) during backward pass. FSDP performs an All-Gather in forward ($1 \times \text{Params}$) and a Reduce-Scatter in backward ($1 \times \text{Params}$) plus another All-Gather in backward, totaling $3 \times \text{Params}$ ($50\%$ more communication).